# Exploration et debug de la base de données SQL

In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect("results.db", timeout=60)
conn.execute("PRAGMA busy_timeout = 60000")  # évite "database is locked"
cur = conn.cursor()

## Présence de compétition par équipe

In [3]:
cur.execute(
    "SELECT DISTINCT course_id, type_code FROM resultat WHERE type_code NOT IN ('INV', 'K1H', 'K1D', 'C1H', 'C1D', 'C2H', 'C2D', 'C2M') ORDER BY course_id;"
)
res = cur.fetchall()
for e in res:
    print(e)

## Double participation sur une course

In [4]:
cur.execute(
    " SELECT r.id, r.id_api, comp.nom AS competition, c.nom AS course, r.course_id,        b.id_api AS bateau, b.libelle AS equipage,        r.bateau_id, r.type_code, r.phase, r.score, r.score_scratch, r.statut FROM resultat r JOIN course c ON c.id = r.course_id JOIN competition comp ON comp.id = c.competition_id JOIN bateau b ON b.id = r.bateau_id JOIN (     SELECT course_id, bateau_id     FROM resultat     GROUP BY course_id, bateau_id     HAVING COUNT(*) > 1 ) d ON d.course_id = r.course_id AND d.bateau_id = r.bateau_id ORDER BY comp.nom, c.nom, r.course_id, r.bateau_id, r.id;"
)
res = cur.fetchall()
for e in res:
    print(e)

## Courses sans résultats

In [5]:
cur.execute(
    " SELECT c.id, c.id_api, c.nom AS course, c.date, co.nom AS competition FROM course c JOIN competition co ON co.id = c.competition_id LEFT JOIN resultat r ON r.course_id = c.id WHERE r.id IS NULL ORDER BY c.date, c.id;"
)
res = cur.fetchall()
for e in res:
    print(e)

## Competiteurs sans résultats

In [6]:
cur.execute(
    " SELECT comp.id, comp.id_api, comp.nom, comp.prenom FROM competiteur comp WHERE NOT EXISTS (     SELECT 1     FROM bateau_competiteur bc     JOIN resultat r ON r.bateau_id = bc.bateau_id     WHERE bc.competiteur_id = comp.id) ORDER BY comp.nom, comp.prenom, comp.id;"
)
res = cur.fetchall()
for e in res:
    print(e)